# CFTC Commitments of Traders on WRDS

Table `trdstrm.dsfutcotrepval`, one row per week and report type. The report type is the last four
characters of `dsmnem`: category (`CM NC NR` legacy | `PM SW MM OR` disaggregated | `TOT`), side
(`L` long, `S` short, `X` spreading), basis (`F` futures-only, `C` combined). So `NCLMMLF` is NYMEX CL,
Managed Money Long, Futures-only. WTI crude `CL` is `contrcode` 1986; the disaggregated history starts
June 2006.

In [1]:
import pandas as pd

try:
    d = pd.read_parquet("cot_raw.parquet")
except FileNotFoundError:
    import wrds
    conn = wrds.Connection(wrds_username="cpehlivanian")
    d = conn.raw_sql("""SELECT date_, dsmnem, numtrades, openinterest
                        FROM trdstrm.dsfutcotrepval
                        WHERE contrcode = 1986 AND date_ >= '2006-06-13'""",
                     date_cols=["date_"])
    conn.close()
    d.to_parquet("cot_raw.parquet")
d.head()

,date_,dsmnem,numtrades,openinterest
0,2006-06-13,NCLCMLC,92.0,924430.0
1,2006-06-20,NCLCMLC,83.0,865700.0
2,2006-06-27,NCLCMLC,87.0,859989.0
3,2006-07-03,NCLCMLC,89.0,864384.0
4,2006-07-11,NCLCMLC,91.0,884493.0


## Pivot to one row per week, one column per report type

A few weeks (the 2018–19 government-shutdown catch-up releases) are missing single fields; carry the
prior week forward for those.

In [2]:
d["rep"] = d["dsmnem"].str[-4:]                                    # NCLMMLF -> MMLF
cot = d.pivot_table(index="date_", columns="rep", values="openinterest").ffill(limit=1)
print(cot.shape)
cot.tail(3).T

(1056, 42)


date_,2026-08-25,2026-09-01,2026-09-08
rep,,,
CMLC,1064393.0,1075159.0,1086716.0
CMLF,892144.0,885135.0,889086.0
CMSC,1284572.0,1300979.0,1323141.0
CMSF,1048390.0,1041140.0,1055271.0
MMLC,203162.0,211013.0,226753.0
MMLF,196882.0,205300.0,218960.0
MMSC,98589.0,91394.0,87414.0
MMSF,112862.0,111019.0,107229.0
MMXC,404055.0,424861.0,417983.0


## The accounting identity

Every contract has exactly one long and one short, so each week

$$\text{OI} = \text{CM}_L + \text{NC}_L + \text{NC}_X + \text{NR}_L
            = \text{CM}_S + \text{NC}_S + \text{NC}_X + \text{NR}_S ,$$

and likewise for the disaggregated categories (PM, SW, MM, OR, NR). Build each side of the identity as
a column, then the residual against `TOTF`.

In [3]:
cot["legacy_long"]  = cot[["CMLF", "NCLF", "NCXF", "NRLF"]].sum(axis=1)
cot["legacy_short"] = cot[["CMSF", "NCSF", "NCXF", "NRSF"]].sum(axis=1)
cot["disagg_long"]  = cot[["PMLF", "SWLF", "SWXF", "MMLF", "MMXF", "ORLF", "ORXF", "NRLF"]].sum(axis=1)
cot["disagg_short"] = cot[["PMSF", "SWSF", "SWXF", "MMSF", "MMXF", "ORSF", "ORXF", "NRSF"]].sum(axis=1)

sides = ["legacy_long", "legacy_short", "disagg_long", "disagg_short"]
for s in sides:
    cot[f"{s}_resid"] = cot[s] - cot["TOTF"]

cot[["TOTF"] + sides + [f"{s}_resid" for s in sides]].tail()

rep,TOTF,legacy_long,legacy_short,disagg_long,disagg_short,legacy_long_resid,legacy_short_resid,disagg_long_resid,disagg_short_resid
date_,,,,,,,,,
2026-08-11,1892429.0,1892429.0,1892429.0,1892429.0,1892429.0,0.0,0.0,0.0,0.0
2026-08-18,1888960.0,1888960.0,1888960.0,1888960.0,1888960.0,0.0,0.0,0.0,0.0
2026-08-25,1906740.0,1906740.0,1906740.0,1906740.0,1906740.0,0.0,0.0,0.0,0.0
2026-09-01,1921085.0,1921085.0,1921085.0,1921085.0,1921085.0,0.0,0.0,0.0,0.0
2026-09-08,1939911.0,1939911.0,1939911.0,1939911.0,1939911.0,0.0,0.0,0.0,0.0


Fraction of weeks each side holds to within one contract, and a look at the weeks where it doesn't.

In [4]:
resid = cot[[f"{s}_resid" for s in sides]]
print((resid.abs() <= 1).mean().round(3))
resid[(resid.abs() > 1).any(axis=1)]

rep
legacy_long_resid     0.996
legacy_short_resid    0.996
disagg_long_resid     0.993
disagg_short_resid    0.994
dtype: Float64


rep,legacy_long_resid,legacy_short_resid,disagg_long_resid,disagg_short_resid
date_,,,,
2017-04-04,0.0,0.0,0.0,25493.0
2017-07-03,0.0,0.0,-16990.0,0.0
2017-08-29,-19430.0,17549.0,-18066.0,16087.0
2017-12-05,3808.0,5103.0,0.0,0.0
2018-04-03,-40703.0,-43461.0,0.0,0.0
2018-07-31,-10543.0,0.0,-2962.0,0.0
2019-02-26,0.0,0.0,-5698.0,-7191.0
2019-03-26,0.0,0.0,-37546.0,-29685.0
2022-04-19,0.0,3275.0,-184.0,3268.0
